In [10]:
#Import Libraries
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
import os

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report
)

#Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
#Load processed datasets
PROJECT_DIR = Path("/content/drive/MyDrive/Diabetes-Prediction")
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"

X_train = pd.read_csv(ARTIFACTS_DIR / "X_train_8features.csv")
X_test = pd.read_csv(ARTIFACTS_DIR / "X_test_8features.csv")

y_train = pd.read_csv(
    ARTIFACTS_DIR / "y_train.csv"
)["Outcome"]

y_test = pd.read_csv(
    ARTIFACTS_DIR / "y_test.csv"
)["Outcome"]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (16000, 8)
X_test: (4000, 8)
y_train: (16000,)
y_test: (4000,)


In [12]:
#Train Random Forest model
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

print("Random Forest training completed.")
print("Number of trees:", model.n_estimators)

Random Forest training completed.
Number of trees: 100


In [ ]:
#5-Fold Cross-Validation
# Create 5-fold cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Evaluate model using cross-validation
scores = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=['accuracy', 'precision', 'recall', 'f1']
)

# Display average scores
print("5-Fold Cross-Validation Results")

print("Accuracy:", scores['test_accuracy'].mean())
print("Precision:", scores['test_precision'].mean())
print("Recall:", scores['test_recall'].mean())
print("F1-Score:", scores['test_f1'].mean())

In [ ]:
#Make predictions
# Predicted classes: 0 or 1
y_pred = model.predict(X_test)

# Predicted probability of class 1 (Diabetic)
y_proba = model.predict_proba(X_test)[:, 1]

print("First 10 actual labels:")
print(y_test.iloc[:10].to_list())

print("\nFirst 10 predicted labels:")
print(y_pred[:10])

print("\nFirst 10 diabetes probabilities:")
print(np.round(y_proba[:10], 3))

In [ ]:
#Evaluate model performance
accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test, y_pred, zero_division=0
)

recall = recall_score(
    y_test, y_pred, zero_division=0
)

f1 = f1_score(
    y_test, y_pred, zero_division=0
)

balanced_acc = balanced_accuracy_score(y_test, y_pred)

print(f"Accuracy:          {accuracy:.4f}")
print(f"Precision:         {precision:.4f}")
print(f"Recall:            {recall:.4f}")
print(f"F1-score:          {f1:.4f}")
print(f"Balanced accuracy: {balanced_acc:.4f}")

print("\nDetailed classification report:")

print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["Non-diabetic", "Diabetic"],
    digits=4,
    zero_division=0
))

In [ ]:
#Confusion matrix\
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=[0, 1]
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Non-diabetic", "Diabetic"],
    yticklabels=["Non-diabetic", "Diabetic"]
)

plt.xlabel("Predicted label")
plt.ylabel("Actual label")
plt.title("Random Forest - Confusion Matrix")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()

print("True Negatives (TN):", tn)
print("False Positives (FP):", fp)
print("False Negatives (FN):", fn)
print("True Positives (TP):", tp)

specificity = tn / (tn + fp) if (tn + fp) else 0

print(f"Specificity: {specificity:.4f}")

In [ ]:
#Save trained model
#MODEL_PATH = ARTIFACTS_DIR / "diabetes_random_forest.joblib"

MODEL_PATH = '/content/drive/MyDrive/Diabetes-Prediction/artifacts/random forest (8).joblib'

joblib.dump(model, MODEL_PATH)

print("Model saved successfully!")
print("Saved Location:", MODEL_PATH)

In [ ]:
#Save evaluation metrics
metrics = {
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-score": f1,
    "Balanced Accuracy": balanced_acc,
    "Specificity": specificity
}

metrics_df = pd.DataFrame(
    list(metrics.items()),
    columns=["Metric", "Score"]
)

display(metrics_df.round(4))

metrics_df.to_csv(
    ARTIFACTS_DIR / "random_forest_metrics (8).csv",
    index=False
)

print("Evaluation metrics saved.")